# Customer Churn Analysis

Explore customer churn patterns and compare two classification models. The notebook reports both ranking performance and errors at the default classification threshold.

In [ ]:
from pathlib import Path
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

# Resolve the data folder whether Jupyter starts in the repository root or notebooks/.
data_file = Path('data/customer_churn_20000.csv')
if not data_file.exists():
    data_file = Path('../data/customer_churn_20000.csv')

df = pd.read_csv(data_file)
display(df.head())
print('Shape:', df.shape)

In [ ]:
print('Missing values by column:\n', df.isna().sum())
print('Duplicate rows:', df.duplicated().sum())
print('Churn distribution:\n', df['Churn'].value_counts(dropna=False))
print('Churn rate:', round(df['Churn'].mean() * 100, 2), '%')
if df['Churn'].isna().any() or not set(df['Churn'].dropna().unique()).issubset({0, 1}):
    raise ValueError('Churn must contain non-missing binary values (0/1).')
df = df.drop_duplicates().copy()

In [ ]:
contract = df.groupby('Contract').agg(Customers=('Customer_ID', 'count'), Churn_Rate=('Churn', 'mean'))
contract['Churn_Rate'] *= 100
internet = df.groupby('Internet_Service')['Churn'].mean().mul(100).sort_values(ascending=False)
satisfaction = df.groupby('Satisfaction_Score')['Churn'].mean().mul(100)
display(contract.sort_values('Churn_Rate', ascending=False))
display(internet)
display(satisfaction)

In [ ]:
X = df.drop(columns=['Customer_ID', 'Churn'])
y = df['Churn'].astype(int)
categorical = ['Contract', 'Internet_Service']
numeric = [column for column in X.columns if column not in categorical]
preprocessor = ColumnTransformer([
    ('categorical', OneHotEncoder(handle_unknown='ignore'), categorical),
    ('numeric', StandardScaler(), numeric)
])
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, class_weight='balanced'),
    'Random Forest': RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42, n_jobs=-1, class_weight='balanced')
}
rows = []
for name, model in models.items():
    pipeline = Pipeline([('preprocessing', preprocessor), ('model', model)])
    pipeline.fit(X_train, y_train)
    predicted = pipeline.predict(X_test)
    probabilities = pipeline.predict_proba(X_test)[:, 1]
    rows.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, predicted),
        'Precision': precision_score(y_test, predicted, zero_division=0),
        'Recall': recall_score(y_test, predicted, zero_division=0),
        'F1': f1_score(y_test, predicted, zero_division=0),
        'ROC_AUC': roc_auc_score(y_test, probabilities)
    })
    print('\n', name)
    print('Confusion matrix (actual rows; predicted columns):\n', confusion_matrix(y_test, predicted))
    print(classification_report(y_test, predicted, zero_division=0))
results = pd.DataFrame(rows).sort_values('ROC_AUC', ascending=False).reset_index(drop=True)
display(results)

## Interpretation

Review precision and recall together. A high recall can identify more customers who churn, but may also flag more customers who would stay. The classification threshold should be chosen with the cost of missed churn and the cost of retention outreach in mind. Segment differences are associations, not proof of cause.

## Power BI handoff

Useful measures include customer count, churned customers, churn rate and average monthly charges. Compare churn by contract, satisfaction, internet service, tenure and support calls.